# 数据建模

## 环境导入

In [1]:
import pickle
import sys
import os
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('default')
sys.path.append(os.path.dirname(os.path.abspath('.')))

from src.models.model_factory import model_train, reorder_columns
from src.utils.all_tolls import create_sample_data
from src.models.model_tools import model_data_clean, sample_select, model_vars_imp, model_features_rfe_select
from src.data.data_clean import *
from src.models.model_report import *
from src.config.feature_config import NOT_MODEL_TRAIN_FEATURE

In [2]:
# 内存情况监控
import psutil
mem_usage = psutil.virtual_memory()
bytes_to_gb = 1024 ** 3

print(f"已使用内存百分比：{mem_usage.percent}%")
print(f"已使用内存：{mem_usage.used / bytes_to_gb:.2f} GB")
print(f"总内存：{mem_usage.total / bytes_to_gb:.2f} GB")

已使用内存百分比：67.2%
已使用内存：10.54 GB
总内存：15.69 GB


## 常量设置

In [3]:
# ---------—---- 参数配置区 --------------------
need_refresh_output_dir = False
y_col_name = "y_flag"
dt_col_name = 'clean_date'
input_data_path = '../input/yl_to_data_zx.csv'

model_type = "lgb"
# ----------------------------------------------

In [4]:
input_name, _ = os.path.splitext(os.path.basename(input_data_path))
output_path = Path('../output')
    
if need_refresh_output_dir and output_path.exists():
    shutil.rmtree(output_path)
    print(f"已清空目录: {output_path}")
need_refresh_output_dir
output_dir = f'../output/{input_name}'
os.makedirs(output_dir, exist_ok=True)

## 1-数据清洗

### 1-1 数据读取

In [5]:
Y_data = pd.read_csv(input_data_path)
Y_data = data_time_clean(Y_data,dt_col_name)

2026-06-08 17:24:52 | INFO     | 时间变量处理开始
2026-06-08 17:24:57 | INFO     | 	> 待处理时间的列为: clean_date, 处理后时间列为: clean_date, 月份列为: clean_yearmonth
2026-06-08 17:24:57 | INFO     | 时间变量处理结束


In [6]:
# 建模数据清洗并导出
NOT_MODEL_TRAIN_FEATURE.extend([y_col_name, dt_col_name, 'clean_date', 'clean_yearmonth'])
# 缺失率处理
model_data1 = calculate_missing_rate(Y_data ,NOT_MODEL_TRAIN_FEATURE )
# 同值率处理
model_data2 = drop_same_rate_high_column(model_data1 ,NOT_MODEL_TRAIN_FEATURE)
# 空值统一替换
model_data = nan_data_replace(model_data2,[y_col_name])
model_data.to_pickle(f"{output_dir}/[data]_clean_input_data.pkl")

2026-06-08 17:24:57 | INFO     | > 缺失率处理开始
2026-06-08 17:24:57 | INFO     | 	> 原始数据集: 行数 494912, 列数 23
2026-06-08 17:24:57 | INFO     | 	> 删除缺失率大于80.0%的值，共计删除 0 列
2026-06-08 17:24:57 | INFO     | 	> 删除列为: []
2026-06-08 17:24:57 | INFO     | 	> 清洗数据集: 行数 494912, 列数 23
2026-06-08 17:24:57 | INFO     | 缺失率处理结束
2026-06-08 17:24:57 | INFO     | 同值率处理开始
	> 处理中 |██████████████████████████████████████████████████| 100.0% 
2026-06-08 17:24:57 | INFO     | 	> 原始数据集: 行数 494912, 列数 23
2026-06-08 17:24:57 | INFO     | 	> 删除同值率大于80.0%的值, 共计删除 0 列
2026-06-08 17:24:57 | INFO     | 	> 删除列为: []
2026-06-08 17:24:57 | INFO     | 	> 最终数据集为: 行数 494912, 列数 23
2026-06-08 17:24:57 | INFO     | 同值率处理完成


## 2-模型训练

### 2-1 模型训练准备

In [7]:
# output_dir = f'../output/{input_name}/'
# model_data = pd.read_pickle(f"{output_dir}/clean_input_data.pkl")

model_data, model_vars = model_data_clean(model_data,y_col_name,NOT_MODEL_TRAIN_FEATURE)
monthly_y_distribution(model_data, y_col_name)

2026-06-08 17:24:58 | INFO     | 总变量数: 23
2026-06-08 17:24:58 | INFO     | 建模变量数: 19
2026-06-08 17:24:58 | INFO     | 其他变量数: 4
2026-06-08 17:24:58 | INFO     | 其他变量: ['map_key', 'clean_date', 'y_flag', 'clean_yearmonth']


,年月,总样本数,坏样本数,好样本数,样本占比,坏样本率,好样本率,PSI
0,202503,39557,1899,37658,7.99%,4.80%,95.20%,0.000000
1,202504,51466,2622,48844,10.40%,5.09%,94.91%,0.000184
2,202505,55031,2967,52064,11.12%,5.39%,94.61%,0.000723
3,202506,45183,2430,42753,9.13%,5.38%,94.62%,0.000691
4,202507,51450,2309,49141,10.40%,4.49%,95.51%,0.000221
5,202508,51696,1747,49949,10.45%,3.38%,96.62%,0.005200
6,202509,67101,1344,65757,13.56%,2.00%,98.00%,0.025266
7,202510,84271,707,83564,17.03%,0.84%,99.16%,0.070721
8,202511,48260,189,48071,9.75%,0.39%,99.61%,0.112495
9,202512,833,0,833,0.17%,0.00%,100.00%,0.002362


In [8]:
oot_list = ['202511','202512','202601']
model_data, data_info = sample_select(model_data,y_col_name,oot_list)
model_data.to_pickle(f"{output_dir}/[data]_train_input_data.pkl")

2026-06-08 17:24:59 | INFO     | 开始数据分割
2026-06-08 17:24:59 | INFO     | 	> 分割参数: test_size: 0.2, random_state: 100
2026-06-08 17:24:59 | INFO     | 	> oot数据集为: ['202511', '202512', '202601']
2026-06-08 17:25:00 | INFO     | 数据分割完成


### 2-2 初次训练

In [9]:
model_info = model_train(model_data,model_vars,y_col_name,model_type,'bys')
model_info_df = reorder_columns(pd.DataFrame(model_info))
model_info_df

2026-06-08 17:25:01 | INFO     | 开始模型训练
2026-06-08 17:25:01 | INFO     | 	> 模型名称: lgb
2026-06-08 17:25:01 | INFO     | 	> 模型调参方式: bys
2026-06-08 17:25:01 | INFO     | 	> 模型参数范围为: {'n_estimators': [100, 200, 300, 400, 500], 'learning_rate': [0.005, 0.01, 0.05, 0.1, 0.2], 'max_depth': [3, 4, 5, 6, 7], 'min_child_samples': [20, 50, 100, 150, 200], 'subsample': [0.6, 0.7, 0.8, 0.9, 1.0], 'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0], 'reg_alpha': [0, 1, 5, 10, 20], 'reg_lambda': [0, 1, 5, 10], 'max_bin': [100, 150, 200, 250, 300]}
2026-06-08 17:25:01 | INFO     | 	> 入模变量数量为: 19
2026-06-08 17:25:01 | INFO     | 贝叶斯优化：共 10 轮迭代，搜索空间: ['n_estimators_idx', 'learning_rate_idx', 'max_depth_idx', 'min_child_samples_idx', 'subsample_idx', 'colsample_bytree_idx', 'reg_alpha_idx', 'reg_lambda_idx', 'max_bin_idx']

2026-06-08 17:25:01 | INFO     | 第 1 次贝叶斯目标优化
2026-06-08 17:25:04 | INFO     | ============================================================
2026-06-08 17:25:04 | INFO     | 参数配置:
2026-06-08

,params,final_score,is_overfit,train_ks,test_ks,vldt_ks,train_auc,test_auc,vldt_auc,train_test_psi,train_vldt_psi,train_test_vldt_psi,model
0,"{'n_estimators': 100, 'learning_rate': 0.005, ...",0.236780,False,0.208,0.196,0.207,0.644,0.633,0.619,0.0000,0.0010,0.0009,"LGBMClassifier(colsample_bytree=0.6, learning_..."
1,"{'n_estimators': 200, 'learning_rate': 0.005, ...",0.233936,False,0.217,0.201,0.204,0.651,0.632,0.615,0.0001,0.0014,0.0013,"LGBMClassifier(colsample_bytree=0.9, learning_..."
2,"{'n_estimators': 100, 'learning_rate': 0.005, ...",0.233800,False,0.205,0.194,0.204,0.641,0.631,0.617,0.0000,0.0000,0.0000,"LGBMClassifier(colsample_bytree=0.6, learning_..."
3,"{'n_estimators': 100, 'learning_rate': 0.005, ...",0.230920,False,0.208,0.197,0.201,0.643,0.632,0.617,0.0000,0.0000,0.0000,"LGBMClassifier(colsample_bytree=0.6, learning_..."
4,"{'n_estimators': 200, 'learning_rate': 0.005, ...",0.229000,False,0.216,0.199,0.199,0.649,0.632,0.612,0.0000,0.0011,0.0010,"LGBMClassifier(colsample_bytree=0.9, learning_..."
5,"{'n_estimators': 100, 'learning_rate': 0.005, ...",0.225900,False,0.209,0.201,0.196,0.644,0.631,0.617,0.0000,0.0000,0.0000,"LGBMClassifier(colsample_bytree=0.7, learning_..."
6,"{'n_estimators': 500, 'learning_rate': 0.005, ...",0.124216,True,0.241,0.203,0.189,0.667,0.635,0.610,0.0001,0.0014,0.0013,"LGBMClassifier(learning_rate=0.005, max_bin=30..."
7,"{'n_estimators': 200, 'learning_rate': 0.05, '...",0.119916,True,0.264,0.210,0.181,0.684,0.637,0.609,0.0001,0.0085,0.0083,"LGBMClassifier(learning_rate=0.05, max_bin=300..."
8,"{'n_estimators': 200, 'learning_rate': 0.2, 'm...",0.118724,True,0.379,0.191,0.178,0.764,0.629,0.613,0.0004,0.0262,0.0254,"LGBMClassifier(colsample_bytree=0.7, learning_..."
9,"{'n_estimators': 300, 'learning_rate': 0.1, 'm...",0.109640,True,0.244,0.204,0.161,0.673,0.636,0.602,0.0000,0.0093,0.0092,"LGBMClassifier(colsample_bytree=0.9, max_bin=2..."


### 2-3 特征筛选

In [10]:
model_choose = 0    # !!!选择需要的模型编号
result_model_vas = model_vars_imp(model_info[model_choose].get("model"), method="all", rate=0.8)
result_model_parms = model_info[model_choose].get("params")

2026-06-08 17:25:45 | INFO     | 变量的选择为：all
2026-06-08 17:25:45 | INFO     | 模型变量总数为： 19


### 2-4 最终模型

In [11]:
model_info_final = model_train(model_data,result_model_vas,y_col_name,model_type,model_params=result_model_parms)
result_model = model_info_final[0].get("model")

with open(f'{output_dir}/[model]_{input_name}_{model_type}.pkl', 'wb') as f:
    pickle.dump(result_model, f)

2026-06-08 17:25:45 | INFO     | 开始模型训练
2026-06-08 17:25:45 | INFO     | 	> 模型名称: lgb
2026-06-08 17:25:45 | INFO     | 	> 模型调参方式: default
2026-06-08 17:25:45 | INFO     | 	> 模型参数范围为: {'n_estimators': 100, 'learning_rate': 0.005, 'max_depth': 7, 'min_child_samples': 20, 'subsample': 1.0, 'colsample_bytree': 0.6, 'reg_alpha': 5, 'reg_lambda': 0, 'max_bin': 150}
2026-06-08 17:25:45 | INFO     | 	> 入模变量数量为: 19
2026-06-08 17:25:48 | INFO     | ============================================================
2026-06-08 17:25:48 | INFO     | 参数配置:
2026-06-08 17:25:48 | INFO     |    n_estimators: 100
2026-06-08 17:25:48 | INFO     |    learning_rate: 0.005
2026-06-08 17:25:48 | INFO     |    max_depth: 7
2026-06-08 17:25:48 | INFO     |    min_child_samples: 20
2026-06-08 17:25:48 | INFO     |    subsample: 1.0
2026-06-08 17:25:48 | INFO     |    colsample_bytree: 0.6
2026-06-08 17:25:48 | INFO     |    reg_alpha: 5
2026-06-08 17:25:48 | INFO     |    reg_lambda: 0
2026-06-08 17:25:48 | INFO     

## 3-模型报告

In [12]:
clean_in_data = pd.read_pickle(f"{output_dir}/[data]_clean_input_data.pkl")
clean_in_data['probability(1)'] = result_model.predict_proba(clean_in_data[result_model.feature_name_])[:, 1]
clean_in_data.to_csv(f"{output_dir}/[data]_clean_input_data_p1.csv", index=False)

In [13]:
report_info = {
    "模型类型": model_type,
    "数据集名称": input_name,
    "y 标列名": y_col_name,
    "vldt 年月集": ', '.join(oot_list)
}

report = model_report_main(
    model=result_model,
    model_info=report_info,
    model_data=model_data,
    y_flag=y_col_name,
    ym_flag='clean_yearmonth',
    data_flag='samp_type',
    output_dir=output_dir,
    output_excel=f'{output_dir}/模型评估报告.xlsx',
    pkl_path=f'{output_dir}/[model]_{input_name}_{model_type}.pkl'
)

✅ 样本模型综合效果报告已生成
✅ 样本入模变量信息报告已生成
✅ 样本变量分箱_是否单调报告已生成
✅ 模型稳定性报告已生成
✅ 变量相关性分析报告已生成
✅ 变量分箱图表数据已生成
✅ 模型报告已导出至: ../output/yl_to_data_zx/模型评估报告.xlsx
✅ 临时文件已保存至: ../output/yl_to_data_zx\tmp
✅ PMML文件已导出: ../output/yl_to_data_zx/[model]_yl_to_data_zx_lgb.pmml
✅ OLE对象已插入: ../output/yl_to_data_zx/[model]_yl_to_data_zx_lgb.pkl -> B13
✅ OLE对象已插入: ../output/yl_to_data_zx/[model]_yl_to_data_zx_lgb.pmml -> B13
